#03_gold_mart_kpi_executivo.sql

Gold - Mart Executivo (resumo mensal)

##Objetivo
- painel executivo de um dataset único por competência e recortes (uf, segmento) com as 3 frentes juntas (inadimplência + custo + experiência)
- útil para storytelling e BI.

##Contexto

In [0]:
USE CATALOG healthcare_dev;

##Cria tabela mart_kpi_executivo

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.gold.mart_kpi_executivo (
  competencia STRING,
  uf STRING,
  segmento_vinculo STRING,

  vidas BIGINT,

  -- churn admin
  pct_inadimplencia DOUBLE,
  gap_pagamento DECIMAL(18,2),

  -- sinistralidade
  custo_pmpm DOUBLE,
  qtd_eventos BIGINT,

  -- experiência
  protocolos_por_1000_vidas DOUBLE,
  nps_medio DOUBLE,
  http_invalid_rate DOUBLE,

  gold_build_ts TIMESTAMP
)
USING DELTA;

##Constroe dataset gold

In [0]:
INSERT OVERWRITE healthcare_dev.gold.mart_kpi_executivo
SELECT
  c.competencia,
  c.uf,
  c.segmento_vinculo,
  c.vidas_expostas AS vidas,

  c.pct_inadimplencia,
  c.gap_pagamento,

  s.custo_pmpm,
  s.qtd_eventos,

  e.protocolos_por_1000_vidas,
  e.nps_medio,
  e.http_invalid_rate,

  current_timestamp() AS gold_build_ts
FROM healthcare_dev.gold.kpi_churn_admin c
LEFT JOIN (
  SELECT competencia, uf, segmento_vinculo,
         SUM(qtd_eventos) AS qtd_eventos,
         -- custo_pmpm já vem por recorte, mas sinistralidade está por rede/tipo/cid; agregamos aqui
         ROUND(SUM(CAST(custo_total AS DOUBLE)) / NULLIF(MAX(vidas_expostas),0), 4) AS custo_pmpm
  FROM healthcare_dev.gold.kpi_sinistralidade
  GROUP BY competencia, uf, segmento_vinculo
) s
ON c.competencia=s.competencia AND c.uf=s.uf AND c.segmento_vinculo=s.segmento_vinculo
LEFT JOIN healthcare_dev.gold.kpi_experiencia e
ON c.competencia=e.competencia AND c.uf=e.uf AND c.segmento_vinculo=e.segmento_vinculo;